# Historical Task 2 random forest baseline

Curated from the original collaborative project (source blob `b0c43e84918cf8729fe88182c507c119b1f5206b`). Local PBL paths are relative and saved outputs were cleared. The original computation and cell order are retained. The notebook requires the original datasets and research dependencies and has not been rerun. **Review the W&B cells before execution:** they initialize a remote run and upload a dataset; do not execute them without permission. The historical data path denotes a non-undersampled split, while the upload path denotes an undersampled file, so these labels must not be conflated. See [baseline results](../../results/baselines/README.md).


In [ ]:
import sys
sys.path.append('..')

In [ ]:
from pathlib import Path
from src.preprocessing import * 
from src.utils import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import matthews_corrcoef as mcc, make_scorer, ConfusionMatrixDisplay, balanced_accuracy_score, f1_score
from matplotlib import colormaps
import wandb

In [ ]:
train_path = Path('../data/embeddings/embeddings_uniprot_exp_enzymes_c30_flens2-98_exp/embeddings_uniprot_exp_enzymes_c30_flens2-98_exp_train85.h5')
test_path = Path('../data/embeddings/embeddings_uniprot_exp_enzymes_c30_flens2-98_exp/embeddings_uniprot_exp_enzymes_c30_flens2-98_exp_test85.h5')
train_set = create_dataset_from_h5(train_path, train_path.stem, unsqueeze=False)
test_set = create_dataset_from_h5(test_path, test_path.stem, unsqueeze=False)

X_train = train_set.X
y_train = train_set.y
X_test = test_set.X
y_test =test_set.y

In [ ]:
rf = RandomForestClassifier(class_weight='balanced')
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

In [ ]:
wandb.init(project='task2_model_selection', name='rf_v5')

In [ ]:
wandb.save('../data/embeddings/embeddings_uniprot_exp_enzymes_c30_flens2-98_usamp_exp/embeddings_uniprot_exp_enzymes_c30_flens2-98_usamp_exp.h5')

In [ ]:
fig_train = class_distribution_from_dataset(train_set)
fig_test = class_distribution_from_dataset(test_set)

display = ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, cmap=colormaps.get('Blues'))
fig, ax = plt.subplots(figsize=(6, 6))
ax.grid(False)
display.plot(ax=ax, cmap=colormaps.get('Blues'))

cv = cross_validate(rf, X_train, y_train, scoring={'balanced_acc': 'balanced_accuracy', 'macro_f1': 'f1_macro', 'mc': make_scorer(mcc)})
cv_table = wandb.Table(columns=["fold", "balanced_acc", 'macro_f1', "mcc"])

for i, (acc, f1, mc) in enumerate(zip(cv['test_balanced_acc'], cv['test_macro_f1'], cv['test_mc'])):
    cv_table.add_data(i, acc, f1, mc)

mean_balanced_acc = np.mean(cv['test_balanced_acc'])
mean_macro_f1 = np.mean(cv['test_macro_f1'])
mean_mcc = np.mean(cv['test_mc'])

# Test scores
test_balanced_acc = balanced_accuracy_score(y_test, y_pred)
test_macro_f1 = f1_score(y_test, y_pred, average='macro')
test_mcc = mcc(y_test, y_pred)

In [ ]:
wandb.log({'class_distribution_train': wandb.Image(fig_train), 'class_distribution_test': wandb.Image(fig_test), 
           'confusion_matrix': wandb.Image(fig),
           "cross_validation": cv_table,
           'cv_mean_balanced_acc': mean_balanced_acc, 'cv_mean_macro_f1': mean_macro_f1, 'cv_mean_mcc': mean_mcc,
           'test_balanced_acc': test_balanced_acc, 'test_macro_f1': test_macro_f1, 'test_mcc': test_mcc})

In [ ]:
wandb.finish()

In [ ]:
test = [1, 2, 3, 4, 5, 6, 7, 8, 9]
i = 3
sl = slice(i - 3, i + 1)
test[sl]